In [ ]:
from pathlib import Path

def data_dir() -> Path:
    """หาโฟลเดอร์ datasets/ โดยไล่ขึ้นจาก cwd."""
    for p in (Path.cwd(), *Path.cwd().parents):
        if (p / "datasets").exists():
            return p / "datasets"
    raise SystemExit("ไม่พบโฟลเดอร์ datasets/ — เปิด JupyterLab จาก repo root (uv run jupyter lab)")

DATA = data_dir()

# # Exercise — Module 4: Classification
#
# **Dataset (โปรดอ่านก่อนใช้งาน):**
# - *UCI SECOM* — McCann, M. & Johnston, A. (2008). *SECOM* [Data set]. UCI Machine Learning Repository. https://doi.org/10.24432/C54305 (CC BY 4.0) — `datasets/secom.zip`
#   - Source: https://archive.ics.uci.edu/dataset/179/secom
#
# แบบฝึกหัดสองชุด: (A) สำรวจข้อมูลและเทียบสี่โมเดลบนสายผลิตจริง, (B) threshold และ feature importance — การตัดสิน pass/fail ที่เทียบเท่าการตั้ง tolerance limit
#
# โจทย์ทุกข้อใช้ **time-based split** (80% แรกเทรน / 20% ท้ายทดสอบ — ข้อมูลมาเป็นลำดับการผลิต) และ random_state=42 ทุกจุด

---

**เวลาที่คาดหวังสำหรับแบบฝึกหัดนี้:** ~45 นาที (ทำในห้องช่วง lab — เฉลยเปิดเผยหลังจบช่วง)


In [ ]:
import io, zipfile
import polars as pl
import numpy as np
import matplotlib.pyplot as plt
# ฟอนต์ไทยสำหรับป้ายกราฟ — fallback list รองรับทุกแพลตฟอร์ม (macOS/Windows)
plt.rcParams["font.family"] = ["DejaVu Sans", "Noto Sans Thai", "Leelawadee UI", "Tahoma", "Thonburi"]

with zipfile.ZipFile(DATA / "secom.zip") as z:
    feats = pl.read_csv(io.BytesIO(z.read("secom.data")), separator=" ", has_header=False,
                        new_columns=[f"feature_{i}" for i in range(1, 591)],
                        schema_overrides=[pl.Float64] * 590, null_values="NaN")
    lab = pl.read_csv(io.BytesIO(z.read("secom_labels.data")), separator=" ", quote_char='"',
                      has_header=False, new_columns=["label", "timestamp"])

secom = feats.with_columns(lab["label"].alias("label"))
ts = lab["timestamp"].str.strptime(pl.Datetime, format="%d/%m/%Y %H:%M:%S", strict=False)
secom = secom.with_columns(ts.alias("ts"))

# time-based split — สำหรับข้อ A2 เป็นต้นไป
n_train = int(len(secom) * 0.8)
train, test = secom.head(n_train), secom.tail(len(secom) - n_train)

# ## ข้อ A1 — สำรวจข้อมูล: ค่าหายและความไม่สมดุล
#
# รายงานสามอย่างจาก `secom` (ทั้ง dataset):
# 1. จำนวนแถว × จำนวน features
# 2. **NaN density** — ค่าหายรวมกี่เปอร์เซ็นต์ของทุกเซลล์ และ top-5 feature ที่หายหนักสุด (ชื่อ + %)
# 3. label distribution — จำนวน PASS / FAIL และอัตรา fail เป็นเปอร์เซ็นต์

In [ ]:
# TODO
n = len(secom)
miss = ...
total_missing = ...
total_cells = n * 590
print(f"shape = ...")
print(f"ค่าหายรวม ... = ...%")
print(miss.head(5))

n_fail = ...
print(f"PASS = ..., FAIL = ... (...%)")

# **คำใบ้:** `secom.drop("label").null_count()` ให้จำนวน NaN ต่อคอลัมน์ — `.transpose(include_header=True, header_name="feature")` แล้วเรียงด้วย `.sort(..., descending=True)` อัตรา fail = จำนวน label == 1 หารด้วย n

# ## ข้อ A2 — สี่โมเดลในตารางเดียว
#
# เทรนสี่ pipeline (`impute(median) → scale → model`) บน `train` แล้วสร้างตารางเทียบ **accuracy เท่านั้น** (train + test):
#
# 1. `LogisticRegression(class_weight="balanced", max_iter=2000, random_state=42)`
# 2. `SVC(kernel="rbf", class_weight="balanced", random_state=42)` ห่อด้วย `CalibratedClassifierCV(..., ensemble=False)`
# 3. `RandomForestClassifier(n_estimators=300, class_weight="balanced_subsample", random_state=42, n_jobs=-1)`
# 4. `XGBClassifier(n_estimators=300, learning_rate=0.1, max_depth=5, scale_pos_weight=..., random_state=42, eval_metric="logloss")` — ⚠️ map label เป็น 0/1 ก่อน และ scale_pos_weight = n_neg/n_pos ของ train
#
# X ของโมเดล = คอลัมน์ feature 590 ตัวเท่านั้น — ตัด `"label"` **และ `"ts"`** ออก (ts ใช้แค่พิสูจน์ลำดับการผลิต)
#
# แล้วตอบใน markdown: **ทำไมตาราง accuracy นี้จึงหลอกเรา**

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# TODO
X_train = train.drop("label", "ts")
X_test = test.drop("label", "ts")

def make_pipe(model):
    ...

models = {...}

# XGBoost ต้องการ label 0/1
y01_train = (train["label"] == 1).cast(pl.Int64)
y01_test = (test["label"] == 1).cast(pl.Int64)

for name, model in models.items():
    ...

# คำตอบข้อ A2 (ทำไมตาราง accuracy หลอกเรา):

# ## ข้อ A3 — class_weight ทำอะไรกับการจับ fail
#
# เทรน `LogisticRegression(max_iter=2000, random_state=42)` สองตัวบน `train` (pipeline impute+scale เหมือนกัน):
# 1. **แบบไม่ใส่** class_weight (default)
# 2. **แบบใส่** `class_weight="balanced"`
#
# สำหรับแต่ละตัว รายงานบน `test`: accuracy, จำนวนชิ้นที่ตัดสิน FAIL (pred == 1), จำนวน fail ที่จับได้ (จาก 17) และ false alarm (ตัดสิน FAIL ทั้งที่จริง PASS)
# แล้วตอบใน markdown หนึ่งประโยค: class_weight เปลี่ยนอะไร

In [ ]:
# TODO
plain = make_pipe(LogisticRegression(max_iter=2000, random_state=42))
balanced = make_pipe(LogisticRegression(class_weight="balanced", max_iter=2000, random_state=42))

for name, model in [("plain", plain), ("balanced", balanced)]:
    ...

# คำตอบข้อ A3:

# ## ข้อ B1 — เลื่อน threshold: จับ fail เพิ่ม แลกกับตัดของดีทิ้ง
#
# ใช้โมเดล logistic **แบบ balanced** จากข้อ A3 คำนวณ `predict_proba` บน `test` (คอลัมน์ของ fail = index 1)
# แล้วทำตารางสำหรับ threshold 0.1 / 0.3 / 0.5 / 0.7 โดยแต่ละแถวรายงาน:
# - จำนวนชิ้นที่ถูกตัดสิน FAIL (`proba >= t`)
# - จำนวน fail ที่จับได้ (จาก 17)
# - false alarm (ตัดสิน FAIL ทั้งที่จริง PASS)

In [ ]:
# TODO
proba_fail = ...   # balanced.predict_proba(X_test)[:, 1]

thr_rows = []
for thr in [0.1, 0.3, 0.5, 0.7]:
    ...

pl.DataFrame(thr_rows)

# **คำใบ้:** `proba_fail = model.predict_proba(X_test)[:, 1]` แล้วในลูป: `flagged = proba_fail >= thr` — จับได้ = `(flagged & (test["label"] == 1)).sum()`, false alarm = `(flagged & (test["label"] == -1)).sum()`

# ## ข้อ B2 — top-10 feature ที่โมเดลสนใจสุด
#
# จาก RandomForest pipeline ของข้อ A2: ดึง `feature_importances_` ออกมา (ระวังต้องเข้าผ่าน `named_steps["model"]`) เรียงหา top-10 แล้วพล็อตแท่งแนวนอน (`barh`)
# จบด้วย **ประโยคเดียว (ภาษาไทย)** ตีความว่าผลนี้บอกอะไรเกี่ยวกับสายผลิต — และอย่าลืมข้อควรระวัง: importance ≠ causation

In [ ]:
# TODO
rf = ...
imp = ...
top = ...

fig, ax = plt.subplots(figsize=(7, 3.8))
...
plt.show()

# **คำใบ้:** `np.argsort(imp)[::-1][:10]` ให้ index ของ top-10 — ชื่อ feature คือ `f"feature_{i + 1}"` เพราะเราตั้งชื่อคอลัมน์เริ่มที่ 1

# คำตอบข้อ B2 (ประโยคเดียว + ข้อควรระวัง):

# ## ข้อ B3 — โจทย์คิด (ตอบใน markdown)
#
# QC manager ของคุณบอกว่า "ของเสียหลุดไปถึงลูกค้า ชิ้นหนึ่ง แพงกว่าการ rework ของดีที่ถูกตัดทิ้ง 20 เท่า"
# จากตารางในข้อ B1 — คุณจะเลือก threshold กี่ และยอมรับผลข้างเคียงอะไร?
# และถ้าสายผลิตเดียวกันนี้เป็น "ของราคาถูก ตัดทิ้งแพงกว่า rework" ทิศทางของ threshold จะกลับไปทางไหน

# คำตอบข้อ B3: